# 🧠 Radiogenomics Analytics Framework
## MRI → Radiomics → Genomic Prediction (MGMT Methylation)

> **Dataset:** RSNA-MICCAI Brain Tumor Radiogenomic Classification  
> **Task:** Predict MGMT promoter methylation status from multi-modal MRI  
> **Framework:** End-to-end radiogenomics pipeline for research & thesis use  

---

### 🔬 What is Radiogenomics?
Radiogenomics is an interdisciplinary field that studies the **statistical associations between medical imaging features (radiomics)** and **genomic/molecular characteristics** of tumors. Instead of invasive biopsies, we extract quantitative imaging biomarkers and learn to predict molecular status non-invasively.

**Clinical relevance:**  
- MGMT (O6-methylguanine-DNA methyltransferase) promoter methylation is a key prognostic and predictive biomarker in glioblastoma (GBM)
- MGMT-methylated tumors respond better to temozolomide chemotherapy
- Standard detection requires invasive tissue biopsy — imaging-based prediction is highly valuable

### 📋 Pipeline Overview
```
DICOM MRI (4 modalities)
        ↓
  Preprocessing (normalize, resize)
        ↓
  ROI Approximation (center crop + intensity threshold)
        ↓
  Radiomics Feature Extraction (intensity, texture, shape)
        ↓
  Feature Selection (correlation, ANOVA, LASSO, RF importance)
        ↓
  Radiogenomic Association Analysis
        ↓
  Multi-modality Fusion + Modeling
        ↓
  Evaluation & Interpretation
```

---
## A. 🛠️ Setup & Installation

We install required libraries and configure the environment. All libraries are available on Kaggle.

In [ ]:
!ls /kaggle/input/competitions

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION A: SETUP
# ─────────────────────────────────────────────────────────────

# Install pydicom if not available (usually pre-installed on Kaggle)
import subprocess
subprocess.run(['pip', 'install', 'pydicom', '-q'])

# ── Core Libraries ──────────────────────────────────────────
import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Medical Imaging ──────────────────────────────────────────
import pydicom
from pydicom.errors import InvalidDicomError

# ── Image Processing ─────────────────────────────────────────
from skimage import exposure, filters, measure
from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import StandardScaler
from scipy import stats

# ── Machine Learning ─────────────────────────────────────────
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, classification_report
)
from sklearn.pipeline import Pipeline

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ── Styling ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444466',
    'axes.labelcolor': '#e0e0ff',
    'xtick.color': '#b0b0cc',
    'ytick.color': '#b0b0cc',
    'text.color': '#e0e0ff',
    'grid.color': '#2a2a4a',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
})
PALETTE = ['#7b68ee', '#00bcd4', '#ff6b6b', '#4caf50', '#ffa726']

# ── Dataset Path ─────────────────────────────────────────────
BASE_PATH = '/kaggle/input/competitions/rsna-miccai-brain-tumor-radiogenomic-classification'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
MODALITIES  = ['FLAIR', 'T1w', 'T1wCE', 'T2w']

print('✅ Setup complete!')
print(f'📂 Dataset path: {BASE_PATH}')
print(f'🧬 Modalities: {MODALITIES}')

---
## B. 📊 Data Understanding

We load the clinical labels, inspect class distribution, and visualize MRI scans across the four modalities.

In [ ]:
print(BASE_PATH)

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION B.1: LOAD LABELS
# ─────────────────────────────────────────────────────────────

train_df = pd.read_csv(os.path.join(BASE_PATH, 'train_labels.csv'))
print('=== Training Labels ===' )
print(train_df.head(10))
print(f'\nShape: {train_df.shape}')
print(f'\nMGMT label counts:')
print(train_df['MGMT_value'].value_counts())

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION B.2: CLASS DISTRIBUTION
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MGMT Methylation Status — Class Distribution', fontsize=14, fontweight='bold', color='#e0e0ff')

# Bar chart
counts = train_df['MGMT_value'].value_counts()
bars = axes[0].bar(
    ['MGMT Unmethylated (0)', 'MGMT Methylated (1)'],
    counts.values,
    color=PALETTE[:2], edgecolor='white', linewidth=0.5
)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(val), ha='center', va='bottom', fontweight='bold', color='white')
axes[0].set_title('Absolute Count', color='#e0e0ff')
axes[0].set_ylabel('Number of Patients')
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
axes[1].pie(
    counts.values,
    labels=['Unmethylated\n(MGMT−)', 'Methylated\n(MGMT+)'],
    colors=PALETTE[:2],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'color': 'white', 'fontsize': 12}
)
axes[1].set_title('Proportion', color='#e0e0ff')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

methylated_pct = counts[1] / counts.sum() * 100
print(f'\n📊 Class balance: {methylated_pct:.1f}% methylated')
print('💡 Note: Slight class imbalance — we will use stratified cross-validation')

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION B.3: MRI VISUALIZATION PER MODALITY
# ─────────────────────────────────────────────────────────────

def load_dicom_slice(patient_id: str, modality: str, slice_idx: int = None) -> np.ndarray | None:
    """
    Load a single DICOM slice from a given patient and modality.
    Returns a normalized float32 array, or None if loading fails.
    """
    folder = os.path.join(TRAIN_PATH, str(patient_id).zfill(5), modality)
    if not os.path.exists(folder):
        return None
    dcm_files = sorted([f for f in os.listdir(folder) if f.endswith('.dcm')])
    if not dcm_files:
        return None
    # Default: use middle slice for best anatomical content
    idx = slice_idx if slice_idx is not None else len(dcm_files) // 2
    idx = min(idx, len(dcm_files) - 1)
    try:
        dcm = pydicom.dcmread(os.path.join(folder, dcm_files[idx]))
        img = dcm.pixel_array.astype(np.float32)
        # Min-max normalization to [0, 1]
        img_min, img_max = img.min(), img.max()
        if img_max > img_min:
            img = (img - img_min) / (img_max - img_min)
        return img
    except Exception:
        return None


# ── Visualize 2 patients × 4 modalities ──────────────────────
sample_ids = train_df['BraTS21ID'].sample(2, random_state=SEED).values
modality_cmaps = {'FLAIR': 'magma', 'T1w': 'gray', 'T1wCE': 'hot', 'T2w': 'viridis'}

fig, axes = plt.subplots(len(sample_ids), len(MODALITIES), figsize=(18, 5 * len(sample_ids)))
fig.suptitle('Multi-Modal MRI Visualization\n(Each row = 1 patient, each column = 1 MRI modality)',
             fontsize=14, fontweight='bold', color='#e0e0ff', y=1.01)

modality_descriptions = {
    'FLAIR': 'Fluid Attenuated\nInversion Recovery',
    'T1w':   'T1-weighted\n(anatomy)',
    'T1wCE': 'T1w + Contrast\n(tumor enhancement)',
    'T2w':   'T2-weighted\n(edema / CSF)'
}

for row_i, pid in enumerate(sample_ids):
    label = train_df.loc[train_df['BraTS21ID'] == pid, 'MGMT_value'].values[0]
    label_str = 'Methylated ✓' if label == 1 else 'Unmethylated ✗'
    for col_j, mod in enumerate(MODALITIES):
        ax = axes[row_i][col_j]
        img = load_dicom_slice(pid, mod)
        if img is not None:
            ax.imshow(img, cmap=modality_cmaps[mod], aspect='equal')
            ax.set_title(
                f'{mod}\n{modality_descriptions[mod]}',
                color='#e0e0ff', fontsize=10, pad=4
            )
        else:
            ax.text(0.5, 0.5, 'No DICOM', ha='center', va='center',
                    transform=ax.transAxes, color='#888')
        if col_j == 0:
            ax.set_ylabel(f'Patient {pid}\nMGMT: {label_str}',
                         color='#ffa726' if label == 1 else '#7b68ee', fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig('mri_modalities.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📌 Key differences between modalities:')
print('  • FLAIR: Suppresses CSF; highlights perilesional edema & infiltration')
print('  • T1w:   Standard anatomy; low signal in edema')
print('  • T1wCE: Contrast enhances blood-brain-barrier disruption (active tumor core)')
print('  • T2w:   High signal in water/edema; peritumoral zone well visible')

---
## C. 🔧 Image Preprocessing

We build a robust preprocessing pipeline: load DICOM → normalize intensity → resize → stack modalities.

**Why preprocessing matters:**
- DICOM pixel values vary across scanners/protocols — normalization ensures comparability
- Consistent spatial dimensions are required for feature extraction
- Multi-modality stacking enables cross-modal feature fusion

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION C: IMAGE PREPROCESSING
# ─────────────────────────────────────────────────────────────

import cv2  # Available on Kaggle

TARGET_SIZE = (128, 128)  # Balance between speed and spatial detail

def preprocess_slice(img: np.ndarray, target_size: tuple = TARGET_SIZE) -> np.ndarray:
    """
    Full preprocessing chain for a single MRI slice:
    1. Resize to standard dimensions
    2. Z-score normalization (zero mean, unit variance)
    3. Clip to ±3σ to reduce outlier influence
    4. Rescale to [0, 1]
    """
    # 1. Resize
    img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    # 2. Z-score normalization
    mean, std = img_resized.mean(), img_resized.std()
    if std > 0:
        img_z = (img_resized - mean) / std
        # 3. Clip at ±3σ
        img_z = np.clip(img_z, -3, 3)
        # 4. Rescale to [0, 1]
        img_norm = (img_z + 3) / 6.0
    else:
        img_norm = np.zeros_like(img_resized)
    return img_norm.astype(np.float32)


def load_patient_volume(
    patient_id: str,
    modalities: list = MODALITIES,
    n_slices: int = 5
) -> dict:
    """
    Load and preprocess multiple slices from all modalities for one patient.
    
    Args:
        patient_id: BraTS21 patient ID
        modalities: list of MRI modality names
        n_slices: number of central slices to aggregate
    
    Returns:
        dict mapping modality → averaged preprocessed slice (H×W)
    """
    volume = {}
    for mod in modalities:
        folder = os.path.join(TRAIN_PATH, str(patient_id).zfill(5), mod)
        if not os.path.exists(folder):
            volume[mod] = None
            continue
        dcm_files = sorted([f for f in os.listdir(folder) if f.endswith('.dcm')])
        if not dcm_files:
            volume[mod] = None
            continue
        # Central slices contain the most diagnostically relevant tumor anatomy
        mid = len(dcm_files) // 2
        half = n_slices // 2
        slice_indices = range(max(0, mid - half), min(len(dcm_files), mid + half + 1))
        slices = []
        for idx in slice_indices:
            try:
                dcm = pydicom.dcmread(os.path.join(folder, dcm_files[idx]))
                raw = dcm.pixel_array.astype(np.float32)
                slices.append(preprocess_slice(raw))
            except Exception:
                continue
        volume[mod] = np.mean(slices, axis=0) if slices else None
    return volume


# ── Preprocessing Visualization ───────────────────────────────
test_pid = train_df['BraTS21ID'].iloc[0]
raw_img = load_dicom_slice(test_pid, 'FLAIR')
proc_img = preprocess_slice(raw_img) if raw_img is not None else None

if raw_img is not None and proc_img is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Preprocessing Pipeline — Patient {test_pid} (FLAIR)',
                 fontsize=13, fontweight='bold', color='#e0e0ff')

    axes[0].imshow(raw_img, cmap='magma')
    axes[0].set_title('1. Raw DICOM\n(min-max [0,1])', color='#e0e0ff')
    axes[0].axis('off')

    axes[1].imshow(proc_img, cmap='magma')
    axes[1].set_title('2. Z-score + Clip + Resize\n(128×128)', color='#e0e0ff')
    axes[1].axis('off')

    # Histogram comparison
    axes[2].hist(raw_img.flatten(), bins=60, alpha=0.6, color=PALETTE[0],
                 label='Raw', density=True)
    axes[2].hist(proc_img.flatten(), bins=60, alpha=0.6, color=PALETTE[1],
                 label='Preprocessed', density=True)
    axes[2].set_title('3. Intensity Histogram', color='#e0e0ff')
    axes[2].legend()
    axes[2].set_xlabel('Pixel Intensity')
    axes[2].set_ylabel('Density')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('preprocessing.png', dpi=150, bbox_inches='tight')
    plt.show()

print('✅ Preprocessing pipeline ready')

---
## D. 🎯 Tumor Region Approximation (Pseudo-ROI)

**Real radiomics requires expert tumor segmentation** (e.g., BraTS annotations) to compute features only within the lesion volume.

**Our pragmatic approach** uses two complementary strategies:
1. **Center crop** — 60% central region where tumors predominantly occur
2. **Intensity thresholding** — isolate bright hyperintense regions (FLAIR) that likely represent tumor/edema

⚠️ **Limitations:**
- May include normal brain tissue, reducing feature specificity
- Cannot differentiate tumor core from edema without proper segmentation
- Real pipelines use tools like FSL, FreeSurfer, or deep learning segmenters (nnU-Net)

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION D: TUMOR REGION APPROXIMATION
# ─────────────────────────────────────────────────────────────

def get_center_crop(img: np.ndarray, crop_frac: float = 0.6) -> np.ndarray:
    """
    Extract central crop of the image.
    Brain tumors in GBM patients predominantly occur in the central/peri-ventricular region.
    """
    h, w = img.shape
    ch, cw = int(h * crop_frac), int(w * crop_frac)
    top  = (h - ch) // 2
    left = (w - cw) // 2
    return img[top:top+ch, left:left+cw]


def get_intensity_mask(img: np.ndarray, percentile: float = 75.0) -> np.ndarray:
    """
    Binary mask: pixels brighter than the given percentile.
    In FLAIR, tumor/edema appear hyperintense (bright) relative to normal brain.
    Returns mask of same spatial dimensions as input.
    """
    threshold = np.percentile(img, percentile)
    return (img > threshold).astype(np.float32)


def apply_roi(
    img: np.ndarray,
    modality: str = 'FLAIR',
    use_intensity: bool = True
) -> np.ndarray:
    """
    Combined ROI: center crop followed by intensity mask (modality-aware).
    
    - FLAIR, T2w: apply intensity threshold (hyperintense tumor/edema)
    - T1wCE:      apply intensity threshold (enhancing tumor)
    - T1w:        center crop only (less hyperintense contrast)
    """
    cropped = get_center_crop(img)
    if use_intensity and modality in ['FLAIR', 'T2w', 'T1wCE']:
        percentile = 70 if modality == 'T1wCE' else 75
        mask = get_intensity_mask(cropped, percentile)
        return cropped * mask
    return cropped


# ── Visualize ROI approximation ───────────────────────────────
if proc_img is not None:
    center_crop = get_center_crop(proc_img)
    intensity_mask = get_intensity_mask(proc_img)
    roi_img = apply_roi(proc_img, modality='FLAIR')

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle('Tumor Region Approximation Strategies (FLAIR)',
                 fontsize=13, fontweight='bold', color='#e0e0ff')

    titles = ['Original\n(preprocessed)', 'Center Crop\n(60%)',
              'Intensity Mask\n(>75th pct)', 'Combined ROI\n(crop × mask)']
    imgs = [proc_img, center_crop, proc_img * intensity_mask, roi_img]

    for ax, title, im in zip(axes, titles, imgs):
        ax.imshow(im, cmap='magma')
        ax.set_title(title, color='#e0e0ff', fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('roi_approximation.png', dpi=150, bbox_inches='tight')
    plt.show()

print('⚠️  ROI Limitation reminder:')
print('   This is a spatial heuristic. For publication-quality radiomics,')
print('   use provided segmentation masks or automated segmentation (nnU-Net).')

---
## E. 🔬 Radiomics Feature Extraction

We extract three categories of quantitative imaging features from each patient:

| Category | Features | Biological Relevance |
|---|---|---|
| **Intensity** | Mean, SD, skewness, kurtosis, energy, entropy | Tumor cellularity, necrosis, heterogeneity |
| **Texture (GLCM)** | Contrast, correlation, homogeneity, dissimilarity | Tissue heterogeneity, architectural complexity |
| **Shape/Morphology** | Area, perimeter, solidity, eccentricity | Tumor invasiveness, geometry |

We extract these for **each of the 4 modalities**, creating a rich multi-modal feature matrix.

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION E: RADIOMICS FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────

def extract_intensity_features(roi: np.ndarray, prefix: str = '') -> dict:
    """
    First-order intensity statistics.
    
    - Mean:     average brightness → tissue density proxy
    - Std:      intensity spread → intratumoral heterogeneity
    - Skewness: distribution asymmetry → necrosis/enhancement patterns
    - Kurtosis: tail behavior → outlier intensity regions
    - Energy:   sum of squared intensities → overall signal strength
    - Entropy:  information content → textural complexity
    - P25/P75:  quartile intensities → robust range measure
    - IQR:      interquartile range → spread without outliers
    """
    flat = roi.flatten()
    flat = flat[flat > 0]  # Exclude background (masked regions)
    if len(flat) < 10:
        return {f'{prefix}_{k}': np.nan for k in [
            'mean','std','skewness','kurtosis','energy','entropy','p25','p75','iqr']}
    p25, p75 = np.percentile(flat, [25, 75])
    hist, _ = np.histogram(flat, bins=64, density=True)
    hist = hist[hist > 0]
    return {
        f'{prefix}_mean':     float(np.mean(flat)),
        f'{prefix}_std':      float(np.std(flat)),
        f'{prefix}_skewness': float(stats.skew(flat)),
        f'{prefix}_kurtosis': float(stats.kurtosis(flat)),
        f'{prefix}_energy':   float(np.sum(flat ** 2)),
        f'{prefix}_entropy':  float(-np.sum(hist * np.log2(hist + 1e-10))),
        f'{prefix}_p25':      float(p25),
        f'{prefix}_p75':      float(p75),
        f'{prefix}_iqr':      float(p75 - p25),
    }


def extract_glcm_features(roi: np.ndarray, prefix: str = '') -> dict:
    """
    Gray-Level Co-occurrence Matrix (GLCM) texture features.
    GLCM captures spatial relationships between pixel intensity pairs.
    
    - Contrast:      local intensity variation → edge sharpness
    - Correlation:   linear dependency → structural regularity
    - Homogeneity:   closeness to diagonal → texture smoothness
    - Dissimilarity: weighted contrast → local texture variation
    - Energy (ASM):  sum of squared elements → textural uniformity
    """
    # Convert to 8-bit for GLCM (requires integer-valued input)
    img_uint8 = (roi * 255).astype(np.uint8)
    if img_uint8.max() == 0:
        return {f'{prefix}_glcm_{k}': np.nan for k in [
            'contrast','correlation','homogeneity','dissimilarity','energy']}
    # Compute GLCM at 4 orientations → rotation-invariant features via mean
    glcm = graycomatrix(
        img_uint8, distances=[1, 3],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256, symmetric=True, normed=True
    )
    props = ['contrast', 'correlation', 'homogeneity', 'dissimilarity', 'energy']
    return {
        f'{prefix}_glcm_{prop}': float(graycoprops(glcm, prop).mean())
        for prop in props
    }


def extract_shape_features(roi: np.ndarray, prefix: str = '') -> dict:
    """
    Morphological / shape proxy features from the ROI mask.
    These capture the geometric properties of the hyperintense region,
    which proxies the gross tumor shape.
    
    - Area fraction:  proportion of bright pixels → tumor burden proxy
    - Solidity:       convex hull fill → convexity/invasion proxy
    - Eccentricity:   elongation → shape irregularity
    - Perimeter:      boundary length → surface complexity
    - Compactness:    4π×area/perimeter² → circularity
    """
    binary = (roi > roi.mean()).astype(np.uint8)
    total_pixels = binary.size
    area_frac = float(binary.sum() / total_pixels)
    regions = measure.regionprops(binary)
    if not regions:
        return {
            f'{prefix}_area_frac':   area_frac,
            f'{prefix}_solidity':    np.nan,
            f'{prefix}_eccentricity':np.nan,
            f'{prefix}_perimeter':   np.nan,
            f'{prefix}_compactness': np.nan,
        }
    r = max(regions, key=lambda x: x.area)  # Largest connected region
    perimeter = r.perimeter if r.perimeter > 0 else 1
    compactness = (4 * np.pi * r.area) / (perimeter ** 2)
    return {
        f'{prefix}_area_frac':    area_frac,
        f'{prefix}_solidity':     float(r.solidity),
        f'{prefix}_eccentricity': float(r.eccentricity),
        f'{prefix}_perimeter':    float(perimeter),
        f'{prefix}_compactness':  float(compactness),
    }


def extract_all_features(patient_id: str) -> dict:
    """
    Master feature extraction function for one patient.
    Loads all modalities, applies ROI approximation, extracts all feature categories.
    Returns a flat dictionary of all features.
    """
    volume = load_patient_volume(patient_id)
    all_features = {'BraTS21ID': patient_id}
    for mod in MODALITIES:
        img = volume.get(mod)
        if img is None:
            # Fill with NaN for missing modalities
            dummy_prefix = mod
            all_features.update({f'{dummy_prefix}_mean': np.nan})
            continue
        roi = apply_roi(img, modality=mod)
        prefix = mod
        all_features.update(extract_intensity_features(roi, prefix))
        all_features.update(extract_glcm_features(roi, prefix))
        all_features.update(extract_shape_features(roi, prefix))
    return all_features


print('🔬 Feature extraction functions defined.')
print(f'   Features per modality: ~{9 + 5 + 5} = 19')
print(f'   Total features (4 modalities): ~{4 * 19} = 76')

In [ ]:
# ─────────────────────────────────────────────────────────────
# FEATURE EXTRACTION: Run on N=50 patient subset
# ─────────────────────────────────────────────────────────────
# Using 50 patients keeps runtime manageable (<10 min on Kaggle)
# For a full study, process all ~585 training patients

N_PATIENTS = 50  # Increase to None for full dataset

# Stratified sample to maintain class balance
subset_df = train_df.groupby('MGMT_value', group_keys=False).apply(
    lambda x: x.sample(min(N_PATIENTS // 2, len(x)), random_state=SEED)
).reset_index(drop=True)

print(f'📊 Using {len(subset_df)} patients ({subset_df["MGMT_value"].value_counts().to_dict()})')
print('⏳ Extracting radiomics features... (may take 5-10 minutes)')

feature_records = []
failed = []
for i, row in subset_df.iterrows():
    pid = row['BraTS21ID']
    try:
        feats = extract_all_features(pid)
        feats['MGMT_value'] = row['MGMT_value']
        feature_records.append(feats)
        if (len(feature_records)) % 10 == 0:
            print(f'  → Processed {len(feature_records)}/{len(subset_df)} patients')
    except Exception as e:
        failed.append(pid)

feat_df = pd.DataFrame(feature_records)
print(f'\n✅ Feature matrix shape: {feat_df.shape}')
print(f'   Failed: {len(failed)} patients')
feat_df.head(3)

In [ ]:
# ── Feature matrix overview ───────────────────────────────────

# Separate feature columns from metadata
meta_cols = ['BraTS21ID', 'MGMT_value']
feature_cols = [c for c in feat_df.columns if c not in meta_cols]

X_raw = feat_df[feature_cols].copy()
y     = feat_df['MGMT_value'].values

print(f'Feature matrix: {X_raw.shape[0]} patients × {X_raw.shape[1]} features')
print(f'Missing values per feature (top 10):')
print(X_raw.isna().sum().sort_values(ascending=False).head(10))

# Impute NaN with median (robust to outliers)
X_raw = X_raw.fillna(X_raw.median())
print(f'\n✅ After median imputation — NaN count: {X_raw.isna().sum().sum()}')

---
## F. 🔍 Feature Selection

High-dimensional feature spaces cause overfitting. We apply a multi-stage selection pipeline:

1. **Correlation filtering** — remove redundant features (r > 0.90)
2. **ANOVA F-test** — univariate statistical test per feature vs MGMT
3. **Mutual Information** — non-linear dependency measure
4. **LASSO** — L1-regularized logistic regression (sparse selection)
5. **Random Forest importance** — ensemble-based ranking

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION F: FEATURE SELECTION
# ─────────────────────────────────────────────────────────────

from sklearn.preprocessing import StandardScaler

# ── Step 1: Correlation Filtering ────────────────────────────
print('━━ Step 1: Correlation Filtering ━━')
corr_matrix = X_raw.corr().abs()
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
# Remove one feature from each highly correlated pair
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.90)]
X_decorr = X_raw.drop(columns=to_drop)
print(f'  Original features: {X_raw.shape[1]}')
print(f'  Dropped (corr > 0.90): {len(to_drop)}')
print(f'  Remaining: {X_decorr.shape[1]}')

# ── Step 2: Standardize ───────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_decorr)
feature_names_decorr = X_decorr.columns.tolist()

# ── Step 3: ANOVA F-test ──────────────────────────────────────
print('\n━━ Step 2: ANOVA F-test ━━')
anova_selector = SelectKBest(score_func=f_classif, k='all')
anova_selector.fit(X_scaled, y)
f_scores  = anova_selector.scores_
f_pvalues = anova_selector.pvalues_

anova_df = pd.DataFrame({
    'feature': feature_names_decorr,
    'f_score': f_scores,
    'p_value': f_pvalues
}).sort_values('f_score', ascending=False)

sig_anova = anova_df[anova_df['p_value'] < 0.05]
print(f'  Significant features (p < 0.05): {len(sig_anova)}')
print(anova_df.head(10).to_string(index=False))

# ── Step 4: Mutual Information ────────────────────────────────
print('\n━━ Step 3: Mutual Information ━━')
mi_scores = mutual_info_classif(X_scaled, y, random_state=SEED)
mi_df = pd.DataFrame({
    'feature': feature_names_decorr,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)
print(f'  Top MI features:')
print(mi_df.head(10).to_string(index=False))

In [ ]:
# ── Step 5: LASSO Feature Selection ──────────────────────────
print('━━ Step 4: LASSO Regularization ━━')
from sklearn.linear_model import LassoCV

lasso = LassoCV(cv=5, random_state=SEED, max_iter=5000)
lasso.fit(X_scaled, y)
lasso_coefs = pd.Series(np.abs(lasso.coef_), index=feature_names_decorr)
lasso_selected = lasso_coefs[lasso_coefs > 0].sort_values(ascending=False)
print(f'  Features selected by LASSO (coef ≠ 0): {len(lasso_selected)}')
print(lasso_selected.head(15))

# ── Step 6: Random Forest Importance ─────────────────────────
print('\n━━ Step 5: Random Forest Feature Importance ━━')
rf_fs = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_fs.fit(X_scaled, y)
rf_importances = pd.Series(rf_fs.feature_importances_, index=feature_names_decorr)
rf_top = rf_importances.sort_values(ascending=False).head(20)
print(f'  Top 20 Random Forest features:')
print(rf_top)

In [ ]:
# ── Consensus Feature Set ─────────────────────────────────────
# Features that appear in top-20 by at least 2 of 3 methods: ANOVA, MI, RF

TOP_K = 20
anova_top = set(anova_df.head(TOP_K)['feature'].tolist())
mi_top    = set(mi_df.head(TOP_K)['feature'].tolist())
rf_top_set = set(rf_top.index.tolist())
lasso_set  = set(lasso_selected.index.tolist())

# Consensus: appear in ≥2 of 4 methods
from collections import Counter
vote_counter = Counter()
for feat_set in [anova_top, mi_top, rf_top_set, lasso_set]:
    for f in feat_set:
        vote_counter[f] += 1

consensus_features = [f for f, v in vote_counter.items() if v >= 2]
print(f'\n📌 Consensus features (≥2 methods agree): {len(consensus_features)}')
print(consensus_features)

# Final feature matrices
X_selected = X_decorr[consensus_features].values
X_selected = scaler.fit_transform(X_selected)  # Re-scale on final set

# ── Feature Selection Summary Plot ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Feature Selection — Multi-Method Ranking',
             fontsize=14, fontweight='bold', color='#e0e0ff')

# ANOVA
top10 = anova_df.head(15)
axes[0].barh(top10['feature'][::-1], top10['f_score'][::-1], color=PALETTE[0])
axes[0].set_title('ANOVA F-score (Top 15)', color='#e0e0ff')
axes[0].set_xlabel('F-score')
axes[0].grid(axis='x', alpha=0.3)

# Mutual Information
mi_top15 = mi_df.head(15)
axes[1].barh(mi_top15['feature'][::-1], mi_top15['mi_score'][::-1], color=PALETTE[1])
axes[1].set_title('Mutual Information (Top 15)', color='#e0e0ff')
axes[1].set_xlabel('MI Score')
axes[1].grid(axis='x', alpha=0.3)

# RF Importance
rf_top15 = rf_importances.sort_values(ascending=False).head(15)
axes[2].barh(rf_top15.index[::-1], rf_top15.values[::-1], color=PALETTE[2])
axes[2].set_title('Random Forest Importance (Top 15)', color='#e0e0ff')
axes[2].set_xlabel('Importance')
axes[2].grid(axis='x', alpha=0.3)

for ax in axes:
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('feature_selection.png', dpi=150, bbox_inches='tight')
plt.show()

---
## G. 🧬 Radiogenomic Analysis (CRITICAL SECTION)

This section is the heart of radiogenomics: **linking imaging features to genomic/molecular status**.

We perform:
1. **Statistical association testing** — which features correlate with MGMT status?
2. **Multi-modality fusion analysis** — does combining MRI sequences improve prediction?
3. **Feature interpretation** — which features are biologically plausible MGMT markers?

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION G.1: STATISTICAL ASSOCIATION ANALYSIS
# ─────────────────────────────────────────────────────────────

print('═══ RADIOGENOMIC ASSOCIATION ANALYSIS ═══')
print('Testing: which imaging features are statistically associated with MGMT methylation?\n')

results = []
for feat in feature_names_decorr:
    vals = X_decorr[feat].values
    group0 = vals[y == 0]
    group1 = vals[y == 1]
    group0 = group0[~np.isnan(group0)]
    group1 = group1[~np.isnan(group1)]
    if len(group0) < 3 or len(group1) < 3:
        continue
    # Mann-Whitney U test (non-parametric, robust to non-normality)
    stat, pval = stats.mannwhitneyu(group0, group1, alternative='two-sided')
    # Effect size: rank-biserial correlation
    n1, n2 = len(group0), len(group1)
    effect_size = 1 - (2 * stat) / (n1 * n2)
    results.append({
        'feature':     feat,
        'u_statistic': stat,
        'p_value':     pval,
        'effect_size': abs(effect_size),
        'mean_unmeth': group0.mean(),
        'mean_meth':   group1.mean(),
        'delta_mean':  group1.mean() - group0.mean(),
    })

assoc_df = pd.DataFrame(results).sort_values('p_value')

# FDR correction (Benjamini-Hochberg)
from statsmodels.stats.multitest import multipletests
_, fdr_pvals, _, _ = multipletests(assoc_df['p_value'].values, method='fdr_bh')
assoc_df['fdr_pval'] = fdr_pvals
assoc_df['significant'] = assoc_df['fdr_pval'] < 0.05

print('Top 15 features associated with MGMT methylation:')
display_cols = ['feature', 'p_value', 'fdr_pval', 'effect_size', 'mean_unmeth', 'mean_meth', 'significant']
print(assoc_df[display_cols].head(15).to_string(index=False, float_format='{:.4f}'.format))

n_sig = assoc_df['significant'].sum()
print(f'\n🔬 Significant after FDR correction (q < 0.05): {n_sig} features')

In [ ]:
# ── Volcano Plot: Radiogenomic Association ────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Radiogenomic Association: Imaging Features vs MGMT Methylation',
             fontsize=14, fontweight='bold', color='#e0e0ff')

# Volcano Plot
ax = axes[0]
log_p  = -np.log10(assoc_df['p_value'].clip(1e-10))
effect = assoc_df['effect_size']
colors_pt = ['#ff6b6b' if sig else '#444466' for sig in assoc_df['significant']]

ax.scatter(effect, log_p, c=colors_pt, alpha=0.8, s=40, edgecolors='none')
ax.axhline(-np.log10(0.05), color='#ffa726', linestyle='--', alpha=0.7, label='p=0.05')
ax.set_xlabel('Effect Size (rank-biserial |r|)', color='#e0e0ff')
ax.set_ylabel('-log10(p-value)', color='#e0e0ff')
ax.set_title('Volcano Plot — Feature–MGMT Associations', color='#e0e0ff')
ax.legend()
ax.grid(True, alpha=0.3)

# Annotate top 5 features
for _, row in assoc_df.head(5).iterrows():
    feat_name = row['feature'].replace('_', '\n')
    xi = row['effect_size']
    yi = -np.log10(max(row['p_value'], 1e-10))
    ax.annotate(feat_name, (xi, yi), fontsize=7, color='#ffd700',
                xytext=(xi+0.02, yi+0.1), arrowprops=dict(arrowstyle='->', color='#888'))

# Box plots: top 5 significant features
ax2 = axes[1]
top5_feats = assoc_df.head(5)['feature'].tolist()
plot_data_meth  = [X_decorr.loc[y == 1, f].dropna().values for f in top5_feats]
plot_data_unmeth = [X_decorr.loc[y == 0, f].dropna().values for f in top5_feats]

positions_0 = np.arange(len(top5_feats)) * 2
positions_1 = positions_0 + 0.7

bp0 = ax2.boxplot(plot_data_unmeth, positions=positions_0, widths=0.6,
                  patch_artist=True,
                  boxprops=dict(facecolor='#7b68ee', alpha=0.7),
                  medianprops=dict(color='white', linewidth=2),
                  whiskerprops=dict(color='#7b68ee'),
                  capprops=dict(color='#7b68ee'),
                  flierprops=dict(marker='o', color='#7b68ee', alpha=0.3, markersize=3))
bp1 = ax2.boxplot(plot_data_meth, positions=positions_1, widths=0.6,
                  patch_artist=True,
                  boxprops=dict(facecolor='#ff6b6b', alpha=0.7),
                  medianprops=dict(color='white', linewidth=2),
                  whiskerprops=dict(color='#ff6b6b'),
                  capprops=dict(color='#ff6b6b'),
                  flierprops=dict(marker='o', color='#ff6b6b', alpha=0.3, markersize=3))

ax2.set_xticks(positions_0 + 0.35)
ax2.set_xticklabels([f.replace('_', '\n') for f in top5_feats], fontsize=8)
ax2.set_title('Top 5 Features: MGMT− vs MGMT+', color='#e0e0ff')
ax2.set_ylabel('Feature Value (standardized)')
ax2.legend([bp0['boxes'][0], bp1['boxes'][0]], ['MGMT− (unmethylated)', 'MGMT+ (methylated)'],
           loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('radiogenomic_association.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION G.2: MULTI-MODALITY FUSION ANALYSIS
# ─────────────────────────────────────────────────────────────
# Compare predictive power: each modality alone vs all combined

print('═══ MULTI-MODALITY FUSION ANALYSIS ═══')
print('Hypothesis: combining FLAIR+T1w+T1wCE+T2w outperforms any single modality\n')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate_modality_subset(modality_list: list, feat_df: pd.DataFrame, y: np.ndarray,
                               cv=cv, label: str = '') -> dict:
    """
    Evaluate a Logistic Regression model using features from specified modalities only.
    Returns mean ROC-AUC, accuracy from 5-fold CV.
    """
    mod_feats = [c for c in feat_df.columns
                 if any(c.startswith(mod + '_') for mod in modality_list)]
    if not mod_feats:
        return {'label': label, 'auc': np.nan, 'acc': np.nan}
    X_mod = feat_df[mod_feats].fillna(feat_df[mod_feats].median()).values
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=SEED, C=0.1))
    ])
    scores = cross_validate(pipe, X_mod, y, cv=cv,
                            scoring=['roc_auc', 'accuracy'], n_jobs=-1)
    return {
        'label':    label,
        'auc':      scores['test_roc_auc'].mean(),
        'auc_std':  scores['test_roc_auc'].std(),
        'acc':      scores['test_accuracy'].mean(),
        'acc_std':  scores['test_accuracy'].std(),
        'n_feats':  len(mod_feats)
    }

fusion_results = []
for mod in MODALITIES:
    res = evaluate_modality_subset([mod], X_decorr, y, label=f'{mod} only')
    fusion_results.append(res)
    print(f'  {mod:8s}: AUC = {res["auc"]:.3f} ± {res["auc_std"]:.3f}  |  ACC = {res["acc"]:.3f}')

# All modalities combined
res_all = evaluate_modality_subset(MODALITIES, X_decorr, y, label='All Modalities (fusion)')
fusion_results.append(res_all)
print(f'  {"All fused":8s}: AUC = {res_all["auc"]:.3f} ± {res_all["auc_std"]:.3f}  |  ACC = {res_all["acc"]:.3f}')

fusion_df = pd.DataFrame(fusion_results)
print(f'\n🏆 Best single modality: {fusion_df.iloc[:-1].loc[fusion_df.iloc[:-1]["auc"].idxmax(), "label"]}')
print(f'🔀 Multi-modality fusion AUC: {res_all["auc"]:.3f}')

In [ ]:
# ── Fusion Analysis Bar Chart ─────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.arange(len(fusion_df))
bar_colors = PALETTE[:4] + ['#ffd700']
bars = ax.bar(x_pos, fusion_df['auc'], color=bar_colors, alpha=0.85,
              yerr=fusion_df['auc_std'], capsize=5, edgecolor='white', linewidth=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(fusion_df['label'], fontsize=11)
ax.set_ylabel('ROC-AUC (5-fold CV mean ± std)')
ax.set_title('Multi-Modality Fusion Analysis\nSingle Modality vs All-Modality Fusion',
             color='#e0e0ff', fontweight='bold')
ax.set_ylim(0.3, 1.0)
ax.axhline(0.5, color='#888', linestyle='--', alpha=0.6, label='Random baseline')
for bar, val, std in zip(bars, fusion_df['auc'], fusion_df['auc_std']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
            f'{val:.3f}', ha='center', fontsize=10, fontweight='bold', color='white')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('modality_fusion.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📌 Clinical interpretation:')
print('   FLAIR captures perilesional edema — closely linked to tumor infiltration')
print('   T1wCE reveals blood-brain barrier disruption — MGMT links to vascular abnormalities')
print('   Multi-modal fusion leverages complementary biological information')

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION G.3: FEATURE INTERPRETATION
# Biological meaning of top radiogenomic features
# ─────────────────────────────────────────────────────────────

BIOLOGICAL_ANNOTATIONS = {
    'mean':        'Average signal intensity → tissue cellularity, edema volume',
    'entropy':     'Information heterogeneity → intratumoral heterogeneity (ITH), key MGMT correlate',
    'skewness':    'Signal asymmetry → necrotic vs viable tumor fraction',
    'kurtosis':    'Tail behavior → outlier enhancement zones, micro-necrosis',
    'std':         'Signal variance → heterogeneous microenvironment',
    'glcm_contrast':'Textural contrast → architectural irregularity, cellular density variation',
    'glcm_correlation': 'Structural regularity → tissue organization (lower in aggressive GBM)',
    'glcm_homogeneity': 'Textural smoothness → uniform tissue regions',
    'glcm_energy': 'Textural uniformity → homogeneous tumor sub-regions',
    'glcm_dissimilarity': 'Local heterogeneity → boundary irregularity',
    'area_frac':   'Proportion of hyperintense region → tumor/edema burden',
    'solidity':    'Convexity → tumor margin regularity (irregular in MGMT-unmethylated)',
    'eccentricity':'Shape elongation → infiltrative growth pattern',
    'compactness': 'Shape circularity → degree of tumor boundary regularity',
    'energy':      'Total signal energy → absolute tumor signal load',
    'iqr':         'Interquartile intensity range → robust heterogeneity measure',
}

print('═══ BIOLOGICAL INTERPRETATION OF TOP FEATURES ═══\n')
for feat in assoc_df.head(10)['feature']:
    # Extract the core feature type from the prefixed name
    for key, bio in BIOLOGICAL_ANNOTATIONS.items():
        if feat.endswith(key):
            mod = feat.split('_')[0]
            pval = assoc_df.loc[assoc_df['feature'] == feat, 'p_value'].values[0]
            eff  = assoc_df.loc[assoc_df['feature'] == feat, 'effect_size'].values[0]
            print(f'📍 {feat}')
            print(f'   Modality: {mod}  |  p={pval:.4f}  |  effect={eff:.3f}')
            print(f'   Biology:  {bio}\n')
            break

---
## H. 🤖 Predictive Modeling

We compare:
- **Logistic Regression** — interpretable linear model, good baseline
- **Random Forest** — ensemble, captures non-linear interactions
- **Before vs after feature selection** — quantify selection benefit

All evaluated with **5-fold stratified cross-validation** (no data leakage).

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION H: MODELING
# ─────────────────────────────────────────────────────────────

from sklearn.dummy import DummyClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ── Define models ─────────────────────────────────────────────
models = {
    'Dummy (baseline)': DummyClassifier(strategy='stratified', random_state=SEED),
    'Logistic Regression': LogisticRegression(max_iter=2000, C=0.1, random_state=SEED),
    'Random Forest':  RandomForestClassifier(n_estimators=300, max_depth=5,
                                             random_state=SEED, n_jobs=-1),
}

# ── Feature configurations ────────────────────────────────────
feature_configs = {
    'All Features (before selection)': X_scaled,  # Full decorrelated set
    'Selected Features (after selection)': X_selected,  # Consensus set
}

results_table = []

print('═══ MODEL EVALUATION ═══')
print(f'Cross-validation: {cv.n_splits}-fold stratified\n')

for feat_name, X_feat in feature_configs.items():
    print(f'\n── Features: {feat_name} ({X_feat.shape[1]} features) ──')
    for model_name, clf in models.items():
        pipe = Pipeline([('clf', clf)])
        cv_results = cross_validate(
            pipe, X_feat, y, cv=cv,
            scoring=['roc_auc', 'accuracy', 'f1'],
            n_jobs=-1
        )
        auc  = cv_results['test_roc_auc'].mean()
        auc_std = cv_results['test_roc_auc'].std()
        acc  = cv_results['test_accuracy'].mean()
        f1   = cv_results['test_f1'].mean()
        results_table.append({
            'Features': feat_name.split(' (')[0],
            'Model': model_name,
            'AUC': auc,
            'AUC_std': auc_std,
            'Accuracy': acc,
            'F1': f1
        })
        print(f'  {model_name:30s}: AUC={auc:.3f}±{auc_std:.3f}  Acc={acc:.3f}  F1={f1:.3f}')

results_df = pd.DataFrame(results_table)
print('\n✅ Modeling complete')

In [ ]:
# ── Performance Comparison Plot ───────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Comparison: Before vs After Feature Selection',
             fontsize=14, fontweight='bold', color='#e0e0ff')

for ax_i, metric in enumerate(['AUC', 'Accuracy']):
    ax = axes[ax_i]
    pivot = results_df.pivot(index='Model', columns='Features', values=metric)
    x = np.arange(len(pivot.index))
    w = 0.35
    cols = pivot.columns.tolist()
    for j, col in enumerate(cols):
        offset = (j - 0.5) * w
        bars = ax.bar(x + offset, pivot[col], w, label=col,
                     color=PALETTE[j], alpha=0.85, edgecolor='white', linewidth=0.5)
        if metric == 'AUC' and col in results_df['Features'].values:
            stds = results_df[(results_df['Features'] == col)]['AUC_std'].values
            ax.errorbar(x + offset, pivot[col], yerr=stds, fmt='none',
                        ecolor='white', capsize=4, alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel(metric)
    ax.set_title(metric, color='#e0e0ff')
    ax.set_ylim(0.3, 1.05)
    ax.axhline(0.5, color='#888', linestyle='--', alpha=0.4)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Summary Table:')
print(results_df.to_string(index=False, float_format='{:.3f}'.format))

---
## I. 📈 Evaluation

Detailed evaluation of the best model: ROC curve, confusion matrix, and classification report.

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION I: EVALUATION
# ─────────────────────────────────────────────────────────────

# Re-train best model on full data for visualization
# (In practice, evaluation should stay within CV folds)
best_model = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=SEED, n_jobs=-1)
best_model.fit(X_selected, y)

# ── ROC Curve via CV ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Model Evaluation — Random Forest (Selected Features)',
             fontsize=14, fontweight='bold', color='#e0e0ff')

# ROC curves per fold
ax_roc = axes[0]
mean_fpr = np.linspace(0, 1, 100)
tprs, aucs = [], []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_selected, y)):
    clf_fold = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=SEED, n_jobs=-1)
    clf_fold.fit(X_selected[train_idx], y[train_idx])
    y_prob = clf_fold.predict_proba(X_selected[val_idx])[:, 1]
    fpr, tpr, _ = roc_curve(y[val_idx], y_prob)
    auc_val = roc_auc_score(y[val_idx], y_prob)
    aucs.append(auc_val)
    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    ax_roc.plot(fpr, tpr, alpha=0.3, color=PALETTE[fold % len(PALETTE)],
                label=f'Fold {fold+1} (AUC={auc_val:.3f})')

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = np.mean(aucs)
std_auc  = np.std(aucs)
ax_roc.plot(mean_fpr, mean_tpr, color='white', linewidth=2.5,
            label=f'Mean ROC (AUC={mean_auc:.3f}±{std_auc:.3f})')
ax_roc.fill_between(mean_fpr,
                    np.mean(tprs, axis=0) - np.std(tprs, axis=0),
                    np.mean(tprs, axis=0) + np.std(tprs, axis=0),
                    alpha=0.15, color='white')
ax_roc.plot([0,1],[0,1], 'r--', alpha=0.5, label='Random')
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves (5-Fold CV)', color='#e0e0ff')
ax_roc.legend(fontsize=7)
ax_roc.grid(True, alpha=0.3)

# Confusion matrix (on full data for display only)
y_pred = best_model.predict(X_selected)
cm = confusion_matrix(y, y_pred)
cm_disp = ConfusionMatrixDisplay(cm, display_labels=['MGMT−\n(Unmethylated)', 'MGMT+\n(Methylated)'])
cm_disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Confusion Matrix\n(full data, for display)', color='#e0e0ff')
axes[1].set_facecolor('#1a1a2e')

# AUC per fold bar chart
axes[2].bar(range(1, 6), aucs, color=PALETTE[:5], alpha=0.85, edgecolor='white')
axes[2].axhline(mean_auc, color='white', linewidth=2, linestyle='--', label=f'Mean={mean_auc:.3f}')
axes[2].axhline(0.5, color='#ff6b6b', linewidth=1, linestyle=':', alpha=0.7, label='Random=0.5')
axes[2].set_xlabel('CV Fold')
axes[2].set_ylabel('ROC-AUC')
axes[2].set_title('AUC per CV Fold', color='#e0e0ff')
axes[2].set_ylim(0, 1)
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification report
print('\n📋 Classification Report (full data):')
print(classification_report(y, y_pred, target_names=['MGMT− (0)', 'MGMT+ (1)']))
print(f'Mean CV AUC: {mean_auc:.3f} ± {std_auc:.3f}')

---
## J. 🔍 Interpretation & Biological Relevance

Feature importance analysis with biological annotation — connecting imaging patterns to molecular biology.

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION J: INTERPRETATION
# ─────────────────────────────────────────────────────────────

# ── Feature Importance (Random Forest) ───────────────────────
rf_importances_sel = pd.Series(
    best_model.feature_importances_,
    index=consensus_features
).sort_values(ascending=True)

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('Radiogenomics Feature Interpretation Dashboard',
             fontsize=15, fontweight='bold', color='#e0e0ff', y=1.01)

# ── Panel 1: Feature Importance ───────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
colors_bar = []
for f in rf_importances_sel.index:
    if f.startswith('FLAIR'): colors_bar.append(PALETTE[0])
    elif f.startswith('T1w'):  colors_bar.append(PALETTE[1])
    elif f.startswith('T1wCE'):colors_bar.append(PALETTE[2])
    else:                     colors_bar.append(PALETTE[3])

bars = ax1.barh(range(len(rf_importances_sel)), rf_importances_sel.values,
                color=colors_bar, alpha=0.85, edgecolor='none')
ax1.set_yticks(range(len(rf_importances_sel)))
ax1.set_yticklabels([f.replace('_glcm_', '\nGLCM_').replace('_', '\n')
                     for f in rf_importances_sel.index], fontsize=7)
ax1.set_xlabel('Feature Importance')
ax1.set_title('Random Forest Feature Importance\n(selected features only)', color='#e0e0ff', fontsize=10)
# Legend
legend_patches = [
    plt.Rectangle((0,0),1,1, fc=PALETTE[i], label=m)
    for i, m in enumerate(MODALITIES)
]
ax1.legend(handles=legend_patches, loc='lower right', fontsize=7)
ax1.grid(axis='x', alpha=0.3)

# ── Panel 2: Correlation heatmap ──────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
feat_subset = consensus_features[:min(12, len(consensus_features))]
corr_subset = X_decorr[feat_subset].corr()
mask = np.triu(np.ones_like(corr_subset, dtype=bool), k=1)
sns.heatmap(
    corr_subset, ax=ax2, mask=mask,
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 6},
    xticklabels=[f.replace('_', '\n') for f in feat_subset],
    yticklabels=[f.replace('_', '\n') for f in feat_subset],
    cbar_kws={'shrink': 0.7}
)
ax2.set_title('Feature Correlation Matrix\n(selected features)', color='#e0e0ff', fontsize=10)
ax2.tick_params(labelsize=6)

# ── Panel 3: MGMT group comparison radar ──────────────────────
ax3 = fig.add_subplot(gs[1, 0])
top5 = rf_importances_sel.tail(5).index.tolist()
for label, color, ls in [(0, PALETTE[0], '-'), (1, PALETTE[2], '--')]:
    group_means = X_decorr.loc[y == label, top5].mean()
    ax3.plot(range(len(top5)), group_means.values, color=color, linewidth=2,
             linestyle=ls, marker='o', markersize=6,
             label=f'MGMT {"−" if label==0 else "+"} (n={sum(y==label)})')
    ax3.fill_between(range(len(top5)), group_means.values, alpha=0.1, color=color)
ax3.set_xticks(range(len(top5)))
ax3.set_xticklabels([f.replace('_', '\n') for f in top5], fontsize=8)
ax3.set_title('Top 5 Features: MGMT− vs MGMT+\n(mean feature values)', color='#e0e0ff', fontsize=10)
ax3.set_ylabel('Mean Feature Value')
ax3.legend()
ax3.grid(True, alpha=0.3)

# ── Panel 4: Modality contribution pie ───────────────────────
ax4 = fig.add_subplot(gs[1, 1])
mod_importance = {}
for mod in MODALITIES:
    mod_feats_sel = [f for f in consensus_features if f.startswith(mod + '_')]
    mod_importance[mod] = sum(best_model.feature_importances_[consensus_features.index(f)]
                               for f in mod_feats_sel if f in consensus_features)

mods_sorted = sorted(mod_importance, key=mod_importance.get, reverse=True)
ax4.pie(
    [mod_importance[m] for m in mods_sorted],
    labels=mods_sorted,
    colors=PALETTE[:4],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'color': 'white', 'fontsize': 11}
)
ax4.set_title('Modality Contribution to\nRandom Forest Prediction', color='#e0e0ff', fontsize=10)

plt.savefig('interpretation_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION J.2: BIOLOGICAL RELEVANCE SUMMARY
# ─────────────────────────────────────────────────────────────

print('═══════════════════════════════════════════════════════════')
print('  RADIOGENOMICS BIOLOGICAL INTERPRETATION SUMMARY')
print('═══════════════════════════════════════════════════════════\n')

print('''
🧬 MGMT METHYLATION — BIOLOGICAL BACKGROUND
─────────────────────────────────────────────
MGMT (O6-methylguanine-DNA methyltransferase) is a DNA repair enzyme.
When its promoter is methylated (silenced), the repair mechanism is
inactivated, making the tumor more sensitive to alkylating agents
(e.g., temozolomide chemotherapy).

MGMT methylation is present in ~40-50% of GBM patients and correlates
with improved overall survival under standard-of-care chemotherapy.

🔬 WHY IMAGING CAN PREDICT MGMT STATUS
────────────────────────────────────────
1. ENTROPY / HETEROGENEITY:
   MGMT-methylated GBMs tend to exhibit greater FLAIR signal heterogeneity,
   reflecting complex mixtures of viable tumor, necrosis, and edema.
   Literature shows entropy features are among the most discriminative
   radiomics biomarkers for MGMT prediction.

2. TEXTURE (GLCM CONTRAST, DISSIMILARITY):
   MGMT-unmethylated tumors often show more homogeneous enhancement
   (T1wCE) due to uniform disruption of the blood-brain barrier.
   Higher GLCM contrast in MGMT-methylated cases may reflect
   heterogeneous vascular architecture.

3. SHAPE (ECCENTRICITY, SOLIDITY):
   MGMT-unmethylated GBMs are often more infiltrative, leading to
   irregular tumor shapes (lower solidity, higher eccentricity).
   MGMT-methylated tumors may exhibit a more circumscribed morphology.

4. SIGNAL INTENSITY (FLAIR MEAN, T2w MEAN):
   MGMT methylation status influences the tumor's water content and
   cellularity — reflected in T2/FLAIR signal levels. Higher FLAIR
   mean often correlates with peritumoral edema extent.

5. MULTI-MODALITY COMPLEMENTARITY:
   - FLAIR: captures infiltrative peritumoral component
   - T1wCE: reflects active angiogenesis and BBB breakdown
   - T2w:   highlights edema and necrotic regions
   - T1w:   baseline anatomy reference
   Each provides a distinct biological view, hence fusion outperforms
   any single modality.

⚠️  STUDY LIMITATIONS & FUTURE DIRECTIONS
──────────────────────────────────────────
• N=50 is underpowered — full dataset (~585 patients) needed for
  publication-quality conclusions
• Pseudo-ROI (center crop + threshold) introduces noise —
  use BraTS segmentation masks for true lesion-specific radiomics
• Reproducibility: radiomics features depend on scan parameters;
  harmonization (ComBat, normalization) needed for multi-site data
• Deep learning approaches (CNNs, transformers) may outperform
  hand-crafted features on this dataset
• External validation cohort required before clinical translation

📚 KEY REFERENCES
──────────────────
• Kickingereder et al. (2016) — Radiogenomics of GBM: MGMT & MRI
• Aerts et al. (2014) — Decoding the tumor phenotype by noninvasive imaging
• Chang et al. (2018) — Deep learning vs radiomics for MGMT prediction
• Gillies et al. (2016) — Radiomics: images are more than pictures
''')

print('═══════════════════════════════════════════════════════════')

In [ ]:
# ─────────────────────────────────────────────────────────────
# FINAL SUMMARY TABLE
# ─────────────────────────────────────────────────────────────

print('\n🏆 FINAL PERFORMANCE SUMMARY')
print('═' * 72)
summary = results_df.copy()
summary['AUC (mean±std)'] = summary.apply(
    lambda r: f"{r['AUC']:.3f} ± {r['AUC_std']:.3f}", axis=1
)
print(summary[['Features', 'Model', 'AUC (mean±std)', 'Accuracy', 'F1']]
      .to_string(index=False, float_format='{:.3f}'.format))

best = results_df.loc[results_df['AUC'].idxmax()]
print(f'\n🥇 Best configuration:')
print(f'   Model:    {best["Model"]}')
print(f'   Features: {best["Features"]}')
print(f'   AUC:      {best["AUC"]:.3f} ± {best["AUC_std"]:.3f}')
print(f'   Accuracy: {best["Accuracy"]:.3f}')

print('\n📁 Saved outputs:')
for fname in ['class_distribution.png', 'mri_modalities.png', 'preprocessing.png',
              'roi_approximation.png', 'feature_selection.png', 'radiogenomic_association.png',
              'modality_fusion.png', 'model_comparison.png', 'evaluation.png',
              'interpretation_dashboard.png']:
    print(f'   • {fname}')

print('\n✅ Radiogenomics framework complete!')